# Shell CLI Demo

This notebook uses shell magic commands only. It demonstrates the public command-line surface for answer rendering, intent parsing, full JSON payloads, local query-object save/load, and figure export.

The saved `qdata.pkl` is a local PxFquery query object. It can be restored by the CLI and reused for answer, JSON, and figure outputs without rerunning the upstream query.

In [1]:
%%bash
set -euo pipefail

cat > .pxfquery_demo_env.sh <<'SH'
if [ -d "src/pxfquery" ]; then
  PXFQUERY_DEMO_ROOT="$(pwd)"
elif [ -d "../src/pxfquery" ]; then
  PXFQUERY_DEMO_ROOT="$(cd .. && pwd)"
else
  PXFQUERY_DEMO_ROOT="$(pwd)"
fi
if [ -f "$PXFQUERY_DEMO_ROOT/.env" ]; then
  set -a
  . "$PXFQUERY_DEMO_ROOT/.env"
  set +a
elif [ -f "$PXFQUERY_DEMO_ROOT/../.env" ]; then
  set -a
  . "$PXFQUERY_DEMO_ROOT/../.env"
  set +a
elif [ -f "$HOME/.env" ]; then
  set -a
  . "$HOME/.env"
  set +a
fi
export PYTHONPATH="$PXFQUERY_DEMO_ROOT/src${PYTHONPATH:+:$PYTHONPATH}"
export MPLBACKEND="Agg"
SH

source .pxfquery_demo_env.sh
python - <<'PYCODE'
import pxfquery
print("PxFquery", pxfquery.__version__)
PYCODE

QUESTION="In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?"
printf '%s\n' "$QUESTION" > pxfquery_demo_question.txt
rm -f pxfquery_demo_qdata.pkl pxfquery_demo_intent.json pxfquery_demo_query.json pxfquery_demo_loaded.json pxfquery_demo_figures.json
rm -rf pxfquery_cli_figures

PxFquery 0.5.13.dev0


In [2]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh
QUESTION=$(cat pxfquery_demo_question.txt)

echo "Question:"
echo "$QUESTION"
echo
echo "$ python -m pxfquery answer \"$QUESTION\""
python -m pxfquery answer "$QUESTION"

Question:
In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout

?



$ python -m pxfquery answer "In A549 lung cancer cells, what functional programs are changed after E

GFR CRISPR knockout?"


[Parsing] start


...

Answer
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon alpha

 response, EMT-I, interferon/MHC-II (I), adipogenesis, and interferon gamma response, and decreased 

cell cycle (G2/M), MYC targets, E2F targets, G2M checkpoint, and MYC.

Analysis source: pxfq

uery 0.5.13.dev0
Evidence: inspect `answer.tables["route_summary"]` and `answer.tables["route_functi

on_results"]`.
Figures: call `pxf.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.


In [3]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh
QUESTION=$(cat pxfquery_demo_question.txt)

echo "$ python -m pxfquery parse \"$QUESTION\""
python -m pxfquery parse "$QUESTION" > pxfquery_demo_intent.json
python - <<'PYCODE'
import json
from pathlib import Path
payload = json.loads(Path("pxfquery_demo_intent.json").read_text())
for key in ["query_type", "bio_context", "pert_desc", "pert_class", "genetic_modality", "forward_result_scope"]:
    print(f"{key}: {payload.get(key)}")
PYCODE

$ python -m pxfquery parse "In A549 lung cancer cells, what functional programs are changed after EG

FR CRISPR knockout?"


query_type: forward
bio_context: A549 lung cancer cells
pert_desc: EGFR
pert_class: genetic
genetic_

modality: knockout
forward_result_scope: both


In [4]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh
QUESTION=$(cat pxfquery_demo_question.txt)

echo "$ python -m pxfquery save \"$QUESTION\" --output pxfquery_demo_qdata.pkl"
python -m pxfquery save "$QUESTION" --output pxfquery_demo_qdata.pkl

echo
echo "Saved object:"
ls -lh pxfquery_demo_qdata.pkl

$ python -m pxfquery save "In A549 lung cancer cells, what functional programs are changed after EGF

R CRISPR knockout?" --output pxfquery_demo_qdata.pkl


[Parsing] start


[Parsing] done | mode=forward; context=A549 lung cancer cells; perturbation=EGFR; time=1.52s
[Matchi

ng] start


...

[Matrix] done | profiles=6; skipped=0; time=0.32s
[Evidence] start


[Evidence] done | status=ready; time=4.15s


pxfquery_demo_qdata.pkl



Saved object:


-rw-r--r--@ 1 dudu  staff   140K Jun 29 23:44 pxfquery_demo_qdata.pkl


In [5]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh

echo "$ python -m pxfquery load pxfquery_demo_qdata.pkl --answer"
python -m pxfquery load pxfquery_demo_qdata.pkl --answer

$ python -m pxfquery load pxfquery_demo_qdata.pkl --answer


Answer
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon respo

nses (alpha and gamma), EMT-I, and adipogenesis, and decreased cell cycle (G2/M), MYC targets, E2F t

argets, and G2M checkpoint programs.

Analysis source: pxfquery 0.5.13.dev0
Evidence: inspec

t `answer.tables["route_summary"]` and `answer.tables["route_function_results"]`.
Figures: call `pxf

.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.


In [6]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh

echo "$ python -m pxfquery load pxfquery_demo_qdata.pkl --json"
python -m pxfquery load pxfquery_demo_qdata.pkl --json > pxfquery_demo_loaded.json
python - <<'PYCODE'
import json
from pathlib import Path
payload = json.loads(Path("pxfquery_demo_loaded.json").read_text())
print("Top-level payload keys:")
print(list(payload.keys()))
print("\nQuestion text:")
print(payload["text"])
print("\nSaved object fields:")
print(list(payload["uns"].keys()))
print("\nRoute status:")
print(payload["uns"]["route_plan"].get("route_status"))
print("\nEvidence layer keys:")
print(list(payload["uns"]["evidence_dossier"].get("evidence_layer", {}).keys()))
PYCODE

$ python -m pxfquery load pxfquery_demo_qdata.pkl --json


Top-level payload keys:
['schema_version', 'text', 'obs', 'uns']

Question text:
In A549 lung cancer

 cells, what functional programs are changed after EGFR CRISPR knockout?

Saved object fields:
['pro

gress_events', 'intent', 'route_plan', 'route_status', 'execution', 'evidence_dossier', 'result', 'p

ickle_path']

Route status:
routed

Evidence layer keys:
['evidence_grade', 'intent_evidence', 'rout

e_evidence', 'matrix_evidence', 'literature_evidence', 'llm_synthesis']


In [7]:
%%bash
set -euo pipefail
source .pxfquery_demo_env.sh
OUT_DIR="pxfquery_cli_figures"
rm -rf "$OUT_DIR"

echo "$ python -m pxfquery load pxfquery_demo_qdata.pkl --figures-output-dir $OUT_DIR --format png"
python -m pxfquery load pxfquery_demo_qdata.pkl --figures-output-dir "$OUT_DIR" --format png > pxfquery_demo_figures.json
cat pxfquery_demo_figures.json

echo
echo "Generated files:"
find "$OUT_DIR" -maxdepth 1 -type f -name '*.png' | sort

$ python -m pxfquery load pxfquery_demo_qdata.pkl --figures-output-dir pxfquery_cli_figures --format

 png


findfont: Failed to find font weight bold, now using 400.


[
  "pxfquery_cli_figures/pxfquery_01_evidence_match_map.png",
  "pxfquery_cli_figures/pxfquery_02_f

unction_match_heatmap.png",
  "pxfquery_cli_figures/pxfquery_03_function_consensus_bar.png"
]



Generated files:


pxfquery_cli_figures/pxfquery_01_evidence_match_map.png
pxfquery_cli_figures/pxfquery_02_function_ma

tch_heatmap.png
pxfquery_cli_figures/pxfquery_03_function_consensus_bar.png


In [8]:
%%bash
set -euo pipefail
rm -f .pxfquery_demo_env.sh pxfquery_demo_question.txt pxfquery_demo_qdata.pkl pxfquery_demo_intent.json pxfquery_demo_query.json pxfquery_demo_loaded.json pxfquery_demo_figures.json
rm -rf pxfquery_cli_figures

echo "Cleaned generated demo files."

Cleaned generated demo files.
